# 🏆 Capstone 2: Multi-Agent Research System

---

## 📖 The Brief

Build a **multi-agent research system** using LangGraph where:
- A **Planner** breaks the topic into sub-queries
- A **Researcher** gathers information per sub-query
- An **Analyst** synthesizes findings
- A **Writer** produces a structured report
- A **Supervisor** coordinates the entire workflow

**Skills tested**: Chapters 6-8 (Agents, LangGraph, Multi-Agent)

---

## 🎯 Requirements

```
╔══════════════════════════════════════════════════════════════════════╗
║  ✅ Supervisor orchestrates agent workflow                         ║
║  ✅ Planner decomposes research topic into sub-queries             ║
║  ✅ Researcher gathers info (simulated web search)                 ║
║  ✅ Writer produces structured report                              ║
║  ✅ Review cycle: if quality < threshold → re-research             ║
║  ✅ Full LangGraph state machine with conditional edges            ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# !pip install langchain-openai langgraph python-dotenv

## 🧪 Part 1: Define State & Tools

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List, Literal
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# Research State
class ResearchState(TypedDict):
    topic: str
    sub_queries: List[str]
    findings: List[str]
    report: str
    review_score: int
    revision_count: int
    max_revisions: int
    current_agent: str

print("✅ State defined with fields: topic, sub_queries, findings, report, review_score")

## 🧪 Part 2: Agent Nodes

In [ ]:
# Planner: Breaks topic into sub-queries
def planner(state: ResearchState) -> dict:
    print("📋 Planner: Decomposing research topic...")
    response = llm.invoke([
        SystemMessage(content="You are a research planner. Break the topic into 3 specific sub-queries. Return them as a numbered list."),
        HumanMessage(content=f"Research topic: {state['topic']}")
    ])
    queries = [line.strip().lstrip('0123456789.-) ') for line in response.content.split('\n') if line.strip() and any(c.isalpha() for c in line)]
    return {"sub_queries": queries[:3], "current_agent": "researcher"}

# Researcher: Gathers information per sub-query
def researcher(state: ResearchState) -> dict:
    print("🔬 Researcher: Gathering information...")
    findings = []
    for query in state["sub_queries"]:
        response = llm.invoke([
            SystemMessage(content="You are a research assistant. Provide 2-3 key findings with specific details for this query. Be factual and concise."),
            HumanMessage(content=query)
        ])
        findings.append(f"Query: {query}\nFindings: {response.content}")
    return {"findings": findings, "current_agent": "writer"}

# Writer: Produces structured report
def writer(state: ResearchState) -> dict:
    print("✍️ Writer: Composing report...")
    findings_text = "\n\n".join(state["findings"])
    response = llm.invoke([
        SystemMessage(content="You are a report writer. Write a structured research report with: Summary, Key Findings (3-5 bullets), and Conclusion. Be professional and concise."),
        HumanMessage(content=f"Topic: {state['topic']}\n\nResearch findings:\n{findings_text}")
    ])
    return {"report": response.content, "current_agent": "reviewer"}

# Reviewer: Scores the report quality
def reviewer(state: ResearchState) -> dict:
    print("🔍 Reviewer: Evaluating report quality...")
    response = llm.invoke([
        SystemMessage(content="You are a quality reviewer. Score this report 1-10 on completeness, accuracy, and clarity. Respond with ONLY a number."),
        HumanMessage(content=f"Topic: {state['topic']}\n\nReport:\n{state['report']}")
    ])
    try:
        score = int(''.join(c for c in response.content if c.isdigit())[:2])
    except (ValueError, IndexError):
        score = 7  # Default if parsing fails
    score = min(score, 10)
    
    revision_count = state.get("revision_count", 0) + 1
    print(f"   Score: {score}/10 (Revision {revision_count})")
    return {"review_score": score, "revision_count": revision_count}

print("✅ All agent nodes defined: planner, researcher, writer, reviewer")

## 🧪 Part 3: Build the Graph

In [ ]:
def should_revise(state: ResearchState) -> Literal["researcher", "end"]:
    """Route: if score < 7 and under revision limit → re-research."""
    if state["review_score"] < 7 and state["revision_count"] < state.get("max_revisions", 2):
        print("   ↩️ Score too low — sending back for revision")
        return "researcher"
    print("   ✅ Report approved!")
    return "end"

# Build graph
graph = StateGraph(ResearchState)

# Add nodes
graph.add_node("planner", planner)
graph.add_node("researcher", researcher)
graph.add_node("writer", writer)
graph.add_node("reviewer", reviewer)

# Add edges
graph.set_entry_point("planner")
graph.add_edge("planner", "researcher")
graph.add_edge("researcher", "writer")
graph.add_edge("writer", "reviewer")
graph.add_conditional_edges("reviewer", should_revise, {
    "researcher": "researcher",
    "end": END
})

# Compile
research_agent = graph.compile()
print("✅ Research agent graph compiled!")
print("   Flow: Planner → Researcher → Writer → Reviewer → (revise or end)")

## 🚀 Part 4: Run the Research Agent

In [ ]:
# Execute research
print("="*60)
print("🔬 MULTI-AGENT RESEARCH SYSTEM")
print("="*60)

result = research_agent.invoke({
    "topic": "The impact of RAG vs Fine-Tuning for enterprise AI applications",
    "sub_queries": [],
    "findings": [],
    "report": "",
    "review_score": 0,
    "revision_count": 0,
    "max_revisions": 2,
    "current_agent": "planner"
})

print("\n" + "="*60)
print("📄 FINAL REPORT")
print("="*60)
print(result["report"])
print(f"\n📊 Final Score: {result['review_score']}/10")
print(f"🔄 Revisions: {result['revision_count']}")

## 🧪 Part 5: Inspect the Workflow

In [ ]:
print("📋 Sub-queries generated:")
for i, q in enumerate(result["sub_queries"], 1):
    print(f"  {i}. {q}")

print(f"\n📦 Research findings: {len(result['findings'])} entries")
print(f"📝 Report length: {len(result['report'])} characters")
print(f"🔄 Total revisions: {result['revision_count']}")
print(f"⭐ Final quality score: {result['review_score']}/10")

## ✅ Capstone 2 Complete!

```
┌───────────────────────────────────────────────────────────────────┐
│  📌 Built: Planner → Researcher → Writer → Reviewer pipeline    │
├───────────────────────────────────────────────────────────────────┤
│  📌 LangGraph: StateGraph with conditional revision cycle        │
├───────────────────────────────────────────────────────────────────┤
│  📌 Quality gate: Review score < 7 triggers re-research          │
├───────────────────────────────────────────────────────────────────┤
│  📌 Production: Max revision limit prevents infinite loops       │
└───────────────────────────────────────────────────────────────────┘
```

---

### 🏆 Multi-Agent Researcher — Done!
Supervisor + specialists + review cycle = autonomous research. 🔬